# Inspect demonstration rollouts

Per-episode placement IoU for the teleoperated Task 2 corpus
[`hermanprawiro/task2_fixpos_200`](https://huggingface.co/datasets/hermanprawiro/task2_fixpos_200).

IoU is **not** in the LeRobot parquet — it lives in the `task2_extras/episodes_task2.jsonl`
sidecar (`success_suggestion.iou_thermalpad_vs_target_current`), recorded at save time.
This notebook plots every episode, then sets `BEST_EPISODE_IDX` to the highest-IoU demo
among those in the bottom 10% of both planar path length (`Σ√(Δx²+Δy²)`) and yaw path
(`Σ|Δyaw|`).
A later cell integrates each demo's base odometry (`Σ|Δx|`, `Σ|Δy|`, `Σ|Δyaw|`) so you can see how much the base was steered during teleop.

The last cell materialises that episode for the open-loop replay adapter and can
launch a scored eval:

1. `python scripts/prepare_replay_actions.py --episode $BEST_EPISODE_IDX` —
   download `meta/` + `data/` only (no videos) and write
   `outputs/replay/task2_fixpos_200/epNNN_actions.npy`. The replay adapter
   serves that npy one chunk at a time; no GPU / lerobot at serve time.
2. `REPLAY_EP=$BEST_EPISODE_IDX ADAPTER=replay ./scripts/launch_policy.sh a10 eval` —
   serve those actions and run a scored eval against the live sim
   (`eval-stack-up` must already be up). Override with `N=…` for more episodes.

**Kernel:** `lerobot-arena` conda env.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("/home/ubuntu/workspace/camelo-ebim")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from camelo.policy.adapters.replay import DEFAULT_DATASET_DIR, DEFAULT_REPO_ID

# --- knobs ---
REPO_ID = DEFAULT_REPO_ID  # hermanprawiro/task2_fixpos_200
DATASET_DIR = ROOT / DEFAULT_DATASET_DIR
EXTRAS_JSONL = DATASET_DIR / "task2_extras" / "episodes_task2.jsonl"
TOP_K = 15

# Gated hub. Prefer the env (HF_TOKEN / HUGGING_FACE_HUB_TOKEN); do not paste a token here.
HF_TOKEN = os.environ.get("HF_TOKEN") or "XXX"

print(f"python={sys.executable}")
print(f"repo={REPO_ID}\nlocal={DATASET_DIR}\nextras={EXTRAS_JSONL}")
print(f"HF_TOKEN set: {bool(HF_TOKEN)}")

In [ ]:
def download_extras_jsonl(repo_id: str, dest: Path, token: str | None) -> Path:
    """Pull only episodes_task2.jsonl (no videos / parquet)."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and dest.stat().st_size > 0:
        print(f"reusing {dest} ({dest.stat().st_size} bytes)")
        return dest
    from huggingface_hub import hf_hub_download
    from huggingface_hub.errors import GatedRepoError

    print(f"downloading {repo_id}:task2_extras/episodes_task2.jsonl → {dest.parent}")
    try:
        path = hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename="task2_extras/episodes_task2.jsonl",
            local_dir=str(dest.parent.parent),
            token=token,
        )
    except GatedRepoError:
        raise SystemExit(
            f"gated: accept the terms at https://huggingface.co/datasets/{repo_id} "
            "and export HF_TOKEN in this kernel's environment"
        ) from None
    return Path(path)


jsonl_path = download_extras_jsonl(REPO_ID, EXTRAS_JSONL, HF_TOKEN)
print(f"jsonl={jsonl_path}")

In [ ]:
def load_episode_iou(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text().splitlines():
        if not line.strip():
            continue
        rec = json.loads(line)
        suggestion = rec.get("success_suggestion") or {}
        rows.append(
            {
                "episode_index": int(rec["episode_index"]),
                "iou": suggestion.get("iou_thermalpad_vs_target_current"),
                "orientation_correct": suggestion.get("is_orientation_correct"),
                "orientation_case": suggestion.get("orientation_case"),
                "success": rec.get("success"),
                "frames": rec.get("frames"),
                "fps_sim": rec.get("fps_sim"),
            }
        )
    df = pd.DataFrame(rows).sort_values("episode_index").reset_index(drop=True)
    df["iou"] = pd.to_numeric(df["iou"], errors="coerce")
    df["duration_s"] = df["frames"] / df["fps_sim"].replace({0: np.nan})
    return df


df = load_episode_iou(jsonl_path)
n_iou = int(df["iou"].notna().sum())
print(
    f"{len(df)} episodes, {n_iou} with IoU  "
    f"success={int(df['success'].fillna(False).sum())}/{len(df)}  "
    f"orientation_ok={int(df['orientation_correct'].fillna(False).sum())}/{len(df)}"
)
df[["episode_index", "iou", "success", "orientation_correct", "orientation_case", "frames", "duration_s"]].describe(
    include="all"
)

In [ ]:
ious = df["iou"].to_numpy(dtype=float)
finite = np.isfinite(ious)
if not finite.any():
    raise ValueError(f"no IoU values in {jsonl_path}")

best_i = int(np.nanargmax(ious))
BEST_EPISODE_IDX = int(df.loc[best_i, "episode_index"])
BEST_IOU = float(ious[best_i])
best_row = df.loc[best_i]

print(
    f"BEST_EPISODE_IDX = {BEST_EPISODE_IDX}   IoU={BEST_IOU:.4f}  "
    f"success={best_row['success']}  orientation={best_row['orientation_case']}  "
    f"frames={best_row['frames']} ({best_row['duration_s']:.1f} sim-s)"
)
print()
print("replay that demo:")
print(f"  python scripts/prepare_replay_actions.py --episode {BEST_EPISODE_IDX}")
print(
    f"  REPLAY_EP={BEST_EPISODE_IDX} ADAPTER=replay ./scripts/launch_policy.sh a10 eval"
)

top = df.sort_values("iou", ascending=False, na_position="last").head(TOP_K)
top[["episode_index", "iou", "success", "orientation_correct", "orientation_case", "frames", "duration_s"]]

In [ ]:
ok = df["orientation_correct"].fillna(False).to_numpy(dtype=bool)
succ = df["success"].fillna(False).to_numpy(dtype=bool)
colors = np.where(ok, "tab:green", "tab:red")

fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [2.2, 1]})

ax = axes[0]
ax.bar(df["episode_index"], np.nan_to_num(ious, nan=0.0), color=colors, width=0.9, linewidth=0)
ax.axhline(BEST_IOU, color="0.2", ls="--", lw=1.0, label=f"best IoU={BEST_IOU:.3f}")
ax.scatter(
    [BEST_EPISODE_IDX], [BEST_IOU],
    c="gold", s=90, zorder=5, edgecolors="0.15", label=f"ep {BEST_EPISODE_IDX}",
)
ax.set_xlabel("episode_index")
ax.set_ylabel("IoU (thermal pad vs target)")
ax.set_title(f"{REPO_ID} — per-episode IoU  (green=orientation ok, red=not)")
ax.set_xlim(-1, int(df["episode_index"].max()) + 1)
ax.set_ylim(0.0, max(1.0, float(np.nanmax(ious)) * 1.05))
ax.grid(True, axis="y", alpha=0.3)
ax.legend(loc="upper right")

axh = axes[1]
axh.hist(ious[finite], bins=30, color="tab:blue", edgecolor="white")
axh.axvline(BEST_IOU, color="gold", lw=2.0, label=f"best ep {BEST_EPISODE_IDX}")
axh.set_xlabel("IoU")
axh.set_ylabel("episodes")
axh.set_title(
    f"IoU histogram  mean={float(np.nanmean(ious)):.3f}  "
    f"median={float(np.nanmedian(ious)):.3f}  "
    f"success={int(succ.sum())}/{len(df)}"
)
axh.grid(True, axis="y", alpha=0.3)
axh.legend(loc="upper right")

fig.tight_layout()
plt.show()
print(f"BEST_EPISODE_IDX = {BEST_EPISODE_IDX}")

In [ ]:
from camelo import contracts as C
from camelo.control.approach import wrap_angle


def load_demo_base_odom(dataset_dir: Path) -> pd.DataFrame:
    files = sorted((dataset_dir / "data").rglob("*.parquet"))
    if not files:
        raise FileNotFoundError(
            f"no parquet under {dataset_dir}/data — run "
            "scripts/prepare_replay_actions.py (downloads meta+data)"
        )
    parts = [
        pd.read_parquet(path, columns=["episode_index", "frame_index", "observation.state"])
        for path in files
    ]
    return pd.concat(parts, ignore_index=True)


def integrated_base_motion(state_rows: pd.DataFrame) -> dict:
    ordered = state_rows.sort_values("frame_index")
    poses = np.stack(ordered["observation.state"].to_numpy())[:, C.S_BASE_ODOM]
    if len(poses) < 2:
        return {"path_x_m": 0.0, "path_y_m": 0.0, "path_xy_m": 0.0, "path_yaw_rad": 0.0}
    dx = np.diff(poses[:, 0])
    dy = np.diff(poses[:, 1])
    dyaw = np.array([wrap_angle(v) for v in np.diff(poses[:, 2])])
    return {
        "path_x_m": float(np.abs(dx).sum()),
        "path_y_m": float(np.abs(dy).sum()),
        "path_xy_m": float(np.hypot(dx, dy).sum()),
        "path_yaw_rad": float(np.abs(dyaw).sum()),
    }


odom = load_demo_base_odom(DATASET_DIR)
rows = [
    {"episode_index": int(ep), **integrated_base_motion(grp)}
    for ep, grp in odom.groupby("episode_index")
]
motion = pd.DataFrame(rows).sort_values("episode_index")

fig_h, axes_h = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True)
for ax_h, (col, ylabel) in zip(
    axes_h,
    (
        ("path_x_m", "Σ|Δx| (m)"),
        ("path_y_m", "Σ|Δy| (m)"),
        ("path_yaw_rad", "Σ|Δyaw| (rad)"),
    ),
):
    ax_h.bar(motion["episode_index"], motion[col], color="tab:blue", width=0.9, linewidth=0)
    ax_h.set_xlabel("episode_index")
    ax_h.set_ylabel(ylabel)
    ax_h.set_xlim(-1, int(motion["episode_index"].max()) + 1)
    ax_h.grid(True, axis="y", alpha=0.3)
fig_h.suptitle(f"{REPO_ID} — base path length during teleop")
fig_h.tight_layout()
plt.show()
print(
    f"{len(motion)} demos  "
    f"median Σ|Δx|={motion.path_x_m.median()*1000:.1f} mm  "
    f"Σ|Δy|={motion.path_y_m.median()*1000:.1f} mm  "
    f"Σ|Δyaw|={motion.path_yaw_rad.median():.4f} rad"
)
motion.describe()

In [ ]:
# Quiet-base pick: bottom PATH_Q of path_xy AND path_yaw, then max IoU.
PATH_Q = 0.10

scored = motion.merge(
    df[["episode_index", "iou", "success", "orientation_case", "frames"]],
    on="episode_index",
    how="inner",
)
xy_cut = float(scored["path_xy_m"].quantile(PATH_Q))
yaw_cut = float(scored["path_yaw_rad"].quantile(PATH_Q))
in_xy = scored["path_xy_m"] <= xy_cut
in_yaw = scored["path_yaw_rad"] <= yaw_cut
pool = scored.loc[in_xy & in_yaw].copy()
if pool.empty:
    raise ValueError(
        f"no demo in bottom {PATH_Q:.0%} of both "
        f"path_xy ≤ {xy_cut*1000:.1f} mm and path_yaw ≤ {yaw_cut:.4f} rad "
        f"(xy-only={int(in_xy.sum())}, yaw-only={int(in_yaw.sum())})"
    )

best = pool.loc[pool["iou"].idxmax()]
BEST_EPISODE_IDX = int(best["episode_index"])
BEST_IOU = float(best["iou"])

print(
    f"cuts  path_xy ≤ {xy_cut*1000:.1f} mm   path_yaw ≤ {yaw_cut:.4f} rad  "
    f"(q={PATH_Q:.0%})"
)
print(
    f"pool  {len(pool)} demos  "
    f"(xy-only={int(in_xy.sum())}, yaw-only={int(in_yaw.sum())})"
)
print(
    f"BEST_EPISODE_IDX = {BEST_EPISODE_IDX}   IoU={BEST_IOU:.4f}  "
    f"path_xy={best.path_xy_m*1000:.1f} mm  "
    f"path_yaw={best.path_yaw_rad:.4f} rad  "
    f"success={best.success}  orientation={best.orientation_case}"
)
print()
print("replay that demo:")
print(f"  python scripts/prepare_replay_actions.py --episode {BEST_EPISODE_IDX}")
print(
    f"  REPLAY_EP={BEST_EPISODE_IDX} ADAPTER=replay ./scripts/launch_policy.sh a10 eval"
)

fig_p, axes_p = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True)
winner_mask = motion["episode_index"].eq(BEST_EPISODE_IDX)
pool_mask = motion["episode_index"].isin(pool["episode_index"])
for ax_p, (col, ylabel) in zip(
    axes_p,
    (
        ("path_xy_m", "Σ√(Δx²+Δy²) (m)"),
        ("path_x_m", "Σ|Δx| (m)"),
        ("path_yaw_rad", "Σ|Δyaw| (rad)"),
    ),
):
    ax_p.bar(motion["episode_index"], motion[col], color="0.75", width=0.9, linewidth=0)
    ax_p.bar(
        motion.loc[pool_mask, "episode_index"],
        motion.loc[pool_mask, col],
        color="tab:orange",
        width=0.9,
        linewidth=0,
        label="pool",
    )
    ax_p.bar(
        motion.loc[winner_mask, "episode_index"],
        motion.loc[winner_mask, col],
        color="gold",
        width=0.9,
        linewidth=0,
        edgecolor="0.15",
        label=f"ep {BEST_EPISODE_IDX}",
    )
    ax_p.set_xlabel("episode_index")
    ax_p.set_ylabel(ylabel)
    ax_p.set_xlim(-1, int(motion["episode_index"].max()) + 1)
    ax_p.grid(True, axis="y", alpha=0.3)
axes_p[0].legend(loc="upper right")
fig_p.suptitle(f"{REPO_ID} — quiet-base pool (bottom {PATH_Q:.0%} xy ∧ yaw), max IoU")
fig_p.tight_layout()
plt.show()

pool.sort_values("iou", ascending=False)[
    ["episode_index", "iou", "path_xy_m", "path_yaw_rad", "orientation_case", "frames"]
]


In [ ]:
# Replay the winning demonstration.
#
# prepare_replay_actions.py writes the 20-dim action column the replay adapter
# loads. launch_policy.sh a10 eval serves it and scores IoU on the live sim
# (needs eval-stack-up). Set RUN_EVAL True only when the scene is already up.
import subprocess

RUN_PREPARE = True
RUN_EVAL = False
EVAL_N = 1  # scored episodes (launch_policy.sh N=)

prepare_cmd = [
    sys.executable,
    str(ROOT / "scripts/prepare_replay_actions.py"),
    "--episode",
    str(BEST_EPISODE_IDX),
]
eval_env = f"REPLAY_EP={BEST_EPISODE_IDX} ADAPTER=replay N={EVAL_N}"
eval_cmd = f"{eval_env} {ROOT / 'scripts/launch_policy.sh'} a10 eval"

print(
    f"winner BEST_EPISODE_IDX={BEST_EPISODE_IDX}  IoU={BEST_IOU:.4f}\n"
    f"  1. {' '.join(prepare_cmd)}\n"
    f"  2. {eval_cmd}"
)

env = os.environ.copy()
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN
    env.setdefault("HUGGING_FACE_HUB_TOKEN", HF_TOKEN)

if RUN_PREPARE:
    subprocess.run(prepare_cmd, cwd=ROOT, env=env, check=True)
else:
    print("skip prepare (RUN_PREPARE=False)")

if RUN_EVAL:
    subprocess.run(eval_cmd, cwd=ROOT, env=env, shell=True, check=True)
else:
    print("skip eval (RUN_EVAL=False) — set RUN_EVAL=True when the sim stack is up")
